# E-Commerce Delivery Delay Prediction - Business Insights

This notebook explores the dataset from a business performance perspective, identifying geographical and category-based pain points from the point of view of operations and sales.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('ggplot')
sns.set_palette("viridis")

df = pd.read_csv('../data/interim/analytical_dataset.csv')
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])

KeyError: 'order_delivered_customer_date'

## 1. Delay Rates by Customer State

Which states experience the most delays?

In [ ]:
state_delays = df.groupby('customer_state')['is_late'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(x='customer_state', y='is_late', data=state_delays)
plt.title('Delay Probability by Customer State')
plt.ylabel('Proportion of Late Deliveries')
plt.axhline(df['is_late'].mean(), color='red', linestyle='--', label='National Average')
plt.legend()
plt.show()

### Insight
* Certain states (specifically RR, CE, AL, MA) have a significantly higher delay rate than the national average. 
* These geographically remote states in the North and Northeast face structural logistics challenges.

## 2. Category Performance

Are some products inherently harder to deliver on time?

In [ ]:
top_cats = df['product_category'].value_counts().head(15).index
cat_delays = df[df['product_category'].isin(top_cats)].groupby('product_category')['is_late'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(x='is_late', y='product_category', data=cat_delays)
plt.title('Delay Rate for Top 15 Product Categories')
plt.xlabel('Proportion of Late Deliveries')
plt.show()

### Insight
* High-volume categories like 'office_furniture' and 'telephony' show relatively high delay profiles.
* Bulky categories often require different logistics paths which may be more prone to delays. In contrast, 'health_beauty' and 'sports_leisure' show much lower delay probabilities.

## 3. Delay Rates by Seller State

Do some origin states cause more delays from a fulfillment point of view?

In [ ]:
seller_delays = df.groupby('seller_state')['is_late'].agg(['mean', 'count']).sort_values('mean', ascending=False).reset_index()
seller_delays = seller_delays[seller_delays['count'] > 50] # Filter out very low volume seller states

plt.figure(figsize=(12, 6))
sns.barplot(x='seller_state', y='mean', data=seller_delays, palette='magma')
plt.title('Delay Probability by Seller State (Min 50 Orders)')
plt.ylabel('Proportion of Late Deliveries')
plt.axhline(df['is_late'].mean(), color='red', linestyle='--', label='National Average')
plt.legend()
plt.show()

### Insight
* Unlike customer states, the core seller states (SP, MG, SC, PR) generally perform right around or slightly below the national average.
* This implies that while the origin matters slightly, the destination (and the link between origin and destination) is a much stronger driver of delays.

## 4. Payment Types vs Delay

Does the method of payment correlate with logistics performance?

In [ ]:
payment_delays = df.groupby('primary_payment_type')['is_late'].agg(['mean', 'count']).sort_values('count', ascending=False).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

sns.barplot(x='primary_payment_type', y='count', data=payment_delays, alpha=0.5, ax=ax1, color='gray')
sns.pointplot(x='primary_payment_type', y='mean', data=payment_delays, ax=ax2, color='red', linestyles='--')

ax1.set_ylabel('Total Volume (Bars)')
ax2.set_ylabel('Delay Rate (Line)', color='red')
plt.title('Payment Type Volumes and Delay Rates')
plt.show()

### Insight
* Credit card dominates the overall volume. 
* Boleto (a common cash-equivalent ticket in Brazil) shows a noticeably lower average delay rate. This could be because boleto approval takes days, meaning sellers only start the fulfillment process later, setting a later estimated delivery baseline, paradoxically causing them to miss the deadline less frequently.

## 5. Monthly Volume vs Delay Rate (Seasonality)

How does the time of year affect the business's ability to fulfill on time?

In [ ]:
df['purchase_month'] = df['order_purchase_timestamp'].dt.to_period('M')
monthly_stats = df.groupby('purchase_month')['is_late'].agg(['mean', 'count']).reset_index()
monthly_stats['purchase_month'] = monthly_stats['purchase_month'].dt.to_timestamp()

fig, ax1 = plt.subplots(figsize=(14, 6))
ax2 = ax1.twinx()

sns.lineplot(x='purchase_month', y='count', data=monthly_stats, ax=ax1, color='blue', label='Total Orders')
sns.lineplot(x='purchase_month', y='mean', data=monthly_stats, ax=ax2, color='red', label='Delay Rate')

ax1.set_ylabel('Order Volume', color='blue')
ax2.set_ylabel('Delay Rate', color='red')
plt.title('Monthly Order Volume vs Delay Rate')
ax1.tick_params(axis='x', rotation=45)
plt.show()

### Insight
* Order volumes climbed steadily throughout the lifecycle of the dataset. 
* Spikes in delay rate don't perfectly map to volume spikes. There are clear systemic breakdown periods (e.g. early 2018) where the delay rate topped 20%, heavily impacting customer satisfaction.

## 6. Freight to Price Ratio Impact

If shipping is very expensive relative to the item, are couriers faster to avoid churn?

In [ ]:
df['freight_ratio'] = df['total_freight'] / df['total_price']
df['high_freight_flag'] = np.where(df['freight_ratio'] > 0.5, 'High Freight (>50%)', 'Normal Freight')

freight_ratio_delays = df.groupby('high_freight_flag')['is_late'].mean().reset_index()

plt.figure(figsize=(8, 5))
sns.barplot(x='high_freight_flag', y='is_late', data=freight_ratio_delays, palette='Set2')
plt.title('Delay Rate: High vs Normal Freight Ratio')
plt.ylabel('Proportion of Late Deliveries')
plt.show()

### Insight
* Counter-intuitively, orders where the customer paid a massive shipping premium (over 50% of the item cost) are *more* likely to be delayed.
* This is a major business risk. High freight is likely correlated with long distances to remote states, but the business fails to meet estimates exactly when the customer paid the most.

## 7. Estimate Accuracy (Difference in Days)

How wrong are the initial delivery estimates?

In [ ]:
df['days_diff'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
df['days_diff'] = df['days_diff'].fillna(0) # For undelivered items

plt.figure(figsize=(10, 6))
sns.histplot(df['days_diff'], bins=50, kde=True, color='purple')
plt.title('Delivery Date Difference (Negative = Early, Positive = Late)')
plt.xlabel('Days Difference (Actual Delivery - Estimated Delivery)')
plt.axvline(0, color='red', linestyle='--', label='Deadline')
plt.xlim(-40, 40)
plt.legend()
plt.show()

### Insight
* Most orders arrive significantly *early*, usually between 5 to 15 days before the promised deadline. The business generally over-promises on the deadline buffer.
* However, the right tail extending past the red zero-line represents the ~8% of total orders that breached the contract, creating negative CSAT.